# Maia Lite 0.3.0 — Avaliação final do SFT Full-Linear (PyTorch)
**Somente inferência; não treina, não exporta, não depende de Ollama.**

Reconstrói **exatamente** a arquitetura e as quatro projeções LoRA (`qkv`, `proj`, `fc`, `out`) do notebook `Maia_Lite_0_3_0_SFT_Full_Linear_v2_L4.ipynb`. Carrega a base **step 300000** e o adaptador `adapter_latest.pt`, verifica a configuração e testa o modelo com o formato original `Usuário: ...\nAssistente:\n`.

**Execução:** Ambiente GPU → executar células em ordem → smoke test → escolher `PROFILE='core'` ou `'full'` → executar bateria → exportar relatórios. Os caminhos são os mesmos do treinador, ajustáveis abaixo. **Não executa treinamento.**

## 1. Ambiente, caminhos e integridade

In [ ]:
import os, json, math, time, random, hashlib, gc, re, csv, unicodedata
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoTokenizer
from google.colab import drive
drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Selecione um runtime com GPU no Colab.'
ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
BASE_CKPT=ROOT/'training_runs/maia_lite_0_3_0_a100_v3_experimental/latest.pt'
TOKENIZER_DIR=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer')
SFT_DIR=ROOT/'training_runs/maia_lite_0_3_0_sft_full_linear_v2'
ADAPTER_CKPT=SFT_DIR/'adapter_latest.pt'
RESULT_DIR=SFT_DIR/'final_capability_audit'
EXPECTED_BASE_STEP=300000
for name,path in [('base',BASE_CKPT),('adapter',ADAPTER_CKPT)]:
    assert path.is_file(),f'{name} ausente: {path}'
assert TOKENIZER_DIR.is_dir(),f'Tokenizer ausente: {TOKENIZER_DIR}'
print('GPU:',torch.cuda.get_device_name(0));print('Base:',BASE_CKPT);print('Adaptador:',ADAPTER_CKPT)


## 2. Arquitetura original do MaiaGPT — preservada integralmente

In [ ]:
from dataclasses import dataclass
@dataclass
class Config:
    vocab_size:int=50257
    block_size:int=2048
    n_layer:int=36
    n_head:int=16
    n_embd:int=1024
    dropout:float=0.0
    bias:bool=True
cfg=Config()
class Block(nn.Module):
    def __init__(self,c):
        super().__init__()
        d=c.n_embd
        self.ln1=nn.LayerNorm(d)
        self.qkv=nn.Linear(d,3*d,bias=c.bias)
        self.proj=nn.Linear(d,d,bias=c.bias)
        self.ln2=nn.LayerNorm(d)
        self.fc=nn.Linear(d,4*d,bias=c.bias)
        self.out=nn.Linear(4*d,d,bias=c.bias)
        self.nh=c.n_head
    def forward(self,x):
        b,t,d=x.shape
        q,k,v=self.qkv(self.ln1(x)).chunk(3,dim=-1)
        q=q.view(b,t,self.nh,d//self.nh).transpose(1,2)
        k=k.view(b,t,self.nh,d//self.nh).transpose(1,2)
        v=v.view(b,t,self.nh,d//self.nh).transpose(1,2)
        a=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.0)
        x=x+self.proj(a.transpose(1,2).contiguous().view(b,t,d))
        x=x+self.out(F.gelu(self.fc(self.ln2(x)),approximate='tanh'))
        return x
class MaiaGPT(nn.Module):
    def __init__(self,c):
        super().__init__();self.cfg=c
        self.tok=nn.Embedding(c.vocab_size,c.n_embd)
        self.pos=nn.Embedding(c.block_size,c.n_embd)
        self.blocks=nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.norm=nn.LayerNorm(c.n_embd)
        self.head=nn.Linear(c.n_embd,c.vocab_size,bias=False)
        self.head.weight=self.tok.weight
        self.apply(self.init_weights)
    @staticmethod
    def init_weights(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,mean=0,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None:nn.init.zeros_(m.bias)
    def forward(self,idx,targets=None):
        b,t=idx.shape
        assert t<=self.cfg.block_size
        x=self.tok(idx)+self.pos(torch.arange(t,device=idx.device))
        for block in self.blocks:x=block(x)
        x=self.norm(x)
        logits=self.head(x)
        loss=None if targets is None else F.cross_entropy(logits.reshape(-1,logits.size(-1)).float(),targets.reshape(-1),ignore_index=-100)
        return logits,loss


## 3. Carregamento dos pesos originais (sem otimizador, sem treino)

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(str(TOKENIZER_DIR),local_files_only=True,use_fast=True)
assert len(tokenizer)==cfg.vocab_size and tokenizer.eos_token_id is not None
base_obj=torch.load(BASE_CKPT,map_location='cpu',weights_only=False) # somente arquivo próprio/confiável
assert 'model' in base_obj, 'Checkpoint-base não contém model'
assert int(base_obj.get('step',-1))==EXPECTED_BASE_STEP, f"Base incorreta: step={base_obj.get('step')}; o SFT foi feito no step 300000"
model=MaiaGPT(cfg)
model.load_state_dict(base_obj['model'],strict=True)
del base_obj;gc.collect()
model=model.to(device='cuda',dtype=torch.bfloat16)
for param in model.parameters():param.requires_grad_(False)


## 4. Injetar Full-Linear LoRA e restaurar o checkpoint final

In [ ]:
RANK=16; ALPHA=32; LORA_DROPOUT=0.05
class LoRALinear(nn.Module):
    def __init__(self, base, rank, alpha, dropout):
        super().__init__()
        assert isinstance(base,nn.Linear)
        self.base=base
        self.scale=alpha/rank
        self.dropout=nn.Dropout(dropout)
        self.A=nn.Parameter(torch.empty(rank,base.in_features,device=base.weight.device,dtype=torch.float32))
        self.B=nn.Parameter(torch.zeros(base.out_features,rank,device=base.weight.device,dtype=torch.float32))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))
    def forward(self,x):
        # Base remains BF16; LoRA computation in BF16 autocast, parameters FP32.
        update=F.linear(F.linear(self.dropout(x).float(),self.A),self.B)*self.scale
        return self.base(x)+update.to(x.dtype)

for block in model.blocks:
    block.qkv=LoRALinear(block.qkv,RANK,ALPHA,LORA_DROPOUT)
    block.proj=LoRALinear(block.proj,RANK,ALPHA,LORA_DROPOUT)
    block.fc=LoRALinear(block.fc,RANK,ALPHA,LORA_DROPOUT)
    block.out=LoRALinear(block.out,RANK,ALPHA,LORA_DROPOUT)

saved=torch.load(ADAPTER_CKPT,map_location='cpu',weights_only=False) # arquivo próprio/confiável
assert 'adapter' in saved and 'config' in saved, 'Formato de checkpoint não corresponde ao SFT Full-Linear'
ac=saved['config']
assert int(ac['rank'])==RANK and int(ac['alpha'])==ALPHA
assert ac.get('targets')==['qkv','proj','fc','out'],f'Targets inesperados: {ac.get("targets")}'
assert int(ac.get('base_step',-1))==EXPECTED_BASE_STEP
adapter=saved['adapter']
expected={name for name,_ in model.named_parameters() if name.endswith('.A') or name.endswith('.B')}
assert set(adapter)==expected, f'LoRA incompleto: ausentes={len(expected-set(adapter))}; extras={len(set(adapter)-expected)}'
with torch.no_grad():
    for name,param in model.named_parameters():
        if name in adapter:
            assert param.shape==adapter[name].shape,(name,param.shape,adapter[name].shape)
            param.copy_(adapter[name].to(device=param.device,dtype=param.dtype))
FINAL_UPDATE=int(saved.get('update',-1))
assert FINAL_UPDATE>0
print('Adaptador carregado:',FINAL_UPDATE,'updates |',len(expected),'tensores LoRA')
if FINAL_UPDATE!=5744:print('AVISO: checkpoint não está no update final 5744; verifique se deseja este checkpoint.')
del saved,adapter;gc.collect();model.eval()


## 5. Geração controlada, compatível com o SFT original

In [ ]:
SEED=20261121
MAX_NEW_TOKENS=72
REPETITION_PENALTY=1.10

def set_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)

@torch.inference_mode()
def generate(question,temperature=0.0,max_new_tokens=MAX_NEW_TOKENS,seed=SEED,repetition_penalty=REPETITION_PENALTY):
    set_seed(seed);model.eval()
    prompt='Usuário: '+question+'\nAssistente:\n'
    ids=tokenizer.encode(prompt,add_special_tokens=False)
    if len(ids)>=cfg.block_size:raise ValueError('Prompt excede contexto')
    generated=[];start=time.perf_counter()
    for _ in range(max_new_tokens):
        x=torch.tensor([(ids+generated)[-cfg.block_size:]],device='cuda')
        with torch.autocast('cuda',dtype=torch.bfloat16):
            logits,_=model(x)
        scores=logits[0,-1].float()
        if repetition_penalty!=1:
            for tid in set(generated):
                if scores[tid]>=0:scores[tid]/=repetition_penalty
                else:scores[tid]*=repetition_penalty
        if temperature==0:
            token=int(scores.argmax())
        else:
            probs=torch.softmax(scores/temperature,dim=-1)
            token=int(torch.multinomial(probs,1))
        if token==tokenizer.eos_token_id:break
        generated.append(token)
        decoded=tokenizer.decode(generated,skip_special_tokens=True)
        if any(marker in decoded for marker in ('\nUsuário:','\nAssistente:')):break
    answer=tokenizer.decode(generated,skip_special_tokens=True).split('\nUsuário:')[0].split('\nAssistente:')[0].strip()
    return {'answer':answer,'generated_tokens':len(generated),'seconds':round(time.perf_counter()-start,3)}


## 6. Smoke test: as três perguntas que falharam

In [ ]:
SMOKE=['Qual é a capital do Brasil?','Diga apenas a palavra azul.','Se Ana é mais velha que Bia, quem é mais nova?']
for q in SMOKE:
    out=generate(q,max_new_tokens=48)
    print('P:',q,'\nR:',out['answer'],'\nTokens:',out['generated_tokens'],'Tempo:',out['seconds'],'s\n')


## 7. Bateria de avaliação — questões com gabarito verificável

O conjunto é fixo e versionado no próprio notebook. Para comparações futuras, não edite o gabarito após observar as respostas. `contains` é uma métrica **permissiva** (pode gerar falsos positivos); a revisão humana continua indispensável. Para tarefas de código, as avaliações por substring são apenas diagnósticas, **não** testes de execução.

In [ ]:
TESTS=[{'id': 'factual_001', 'category': 'factual', 'language': 'pt', 'prompt': 'Qual é a capital do Brasil?', 'expected': 'brasília', 'mode': 'contains'}, {'id': 'factual_002', 'category': 'factual', 'language': 'pt', 'prompt': 'Qual é a capital da França?', 'expected': 'paris', 'mode': 'contains'}, {'id': 'factual_003', 'category': 'factual', 'language': 'pt', 'prompt': 'Qual é a capital do Japão?', 'expected': 'tóquio', 'mode': 'contains'}, {'id': 'factual_004', 'category': 'factual', 'language': 'pt', 'prompt': 'Em qual continente fica o Brasil?', 'expected': 'américa do sul', 'mode': 'contains'}, {'id': 'factual_005', 'category': 'factual', 'language': 'pt', 'prompt': 'Quantos dias tem uma semana?', 'expected': '7', 'mode': 'contains'}, {'id': 'factual_006', 'category': 'factual', 'language': 'pt', 'prompt': 'Qual planeta é conhecido como planeta vermelho?', 'expected': 'marte', 'mode': 'contains'}, {'id': 'factual_007', 'category': 'factual', 'language': 'pt', 'prompt': 'Qual é o maior oceano da Terra?', 'expected': 'pacífico', 'mode': 'contains'}, {'id': 'factual_008', 'category': 'factual', 'language': 'pt', 'prompt': 'Qual é o idioma oficial do Brasil?', 'expected': 'português', 'mode': 'contains'}, {'id': 'factual_009', 'category': 'factual', 'language': 'pt', 'prompt': 'Qual é o símbolo químico da água?', 'expected': 'h2o', 'mode': 'contains'}, {'id': 'factual_010', 'category': 'factual', 'language': 'pt', 'prompt': 'Quem escreveu Dom Casmurro?', 'expected': 'machado de assis', 'mode': 'contains'}, {'id': 'instrucao_estrita_011', 'category': 'instrucao_estrita', 'language': 'pt', 'prompt': 'Diga apenas a palavra azul.', 'expected': 'azul', 'mode': 'exact'}, {'id': 'instrucao_estrita_012', 'category': 'instrucao_estrita', 'language': 'pt', 'prompt': 'Responda somente SIM, em maiúsculas.', 'expected': 'SIM', 'mode': 'exact'}, {'id': 'instrucao_estrita_013', 'category': 'instrucao_estrita', 'language': 'pt', 'prompt': 'Escreva apenas o número 42.', 'expected': '42', 'mode': 'exact'}, {'id': 'instrucao_estrita_014', 'category': 'instrucao_estrita', 'language': 'pt', 'prompt': 'Responda somente com a palavra gato.', 'expected': 'gato', 'mode': 'exact'}, {'id': 'instrucao_estrita_015', 'category': 'instrucao_estrita', 'language': 'pt', 'prompt': 'Escreva exatamente: OK', 'expected': 'OK', 'mode': 'exact'}, {'id': 'instrucao_estrita_016', 'category': 'instrucao_estrita', 'language': 'pt', 'prompt': 'Responda apenas não.', 'expected': 'não', 'mode': 'exact'}, {'id': 'instrucao_estrita_017', 'category': 'instrucao_estrita', 'language': 'pt', 'prompt': 'Escreva apenas a letra A.', 'expected': 'A', 'mode': 'exact'}, {'id': 'instrucao_estrita_018', 'category': 'instrucao_estrita', 'language': 'pt', 'prompt': 'Diga somente verde.', 'expected': 'verde', 'mode': 'exact'}, {'id': 'instrucao_estrita_019', 'category': 'instrucao_estrita', 'language': 'pt', 'prompt': 'Responda exatamente com duas letras: PT', 'expected': 'PT', 'mode': 'exact'}, {'id': 'instrucao_estrita_020', 'category': 'instrucao_estrita', 'language': 'pt', 'prompt': 'Retorne apenas o número 7.', 'expected': '7', 'mode': 'exact'}, {'id': 'logica_021', 'category': 'logica', 'language': 'pt', 'prompt': 'Se Ana é mais velha que Bia, quem é mais nova?', 'expected': 'bia', 'mode': 'contains'}, {'id': 'logica_022', 'category': 'logica', 'language': 'pt', 'prompt': 'Se João é mais alto que Pedro, quem é mais baixo?', 'expected': 'pedro', 'mode': 'contains'}, {'id': 'logica_023', 'category': 'logica', 'language': 'pt', 'prompt': 'Maria tem 3 maçãs e ganha 2. Quantas tem?', 'expected': '5', 'mode': 'contains'}, {'id': 'logica_024', 'category': 'logica', 'language': 'pt', 'prompt': 'Se A vem antes de B, qual vem depois?', 'expected': 'b', 'mode': 'contains'}, {'id': 'logica_025', 'category': 'logica', 'language': 'pt', 'prompt': 'Carlos é irmão de Ana. Ana é irmã de Carlos? Responda sim ou não.', 'expected': 'sim', 'mode': 'contains'}, {'id': 'logica_026', 'category': 'logica', 'language': 'pt', 'prompt': 'Um objeto está dentro de uma caixa. Onde está o objeto?', 'expected': 'caixa', 'mode': 'contains'}, {'id': 'logica_027', 'category': 'logica', 'language': 'pt', 'prompt': 'Se hoje é segunda-feira, que dia é amanhã?', 'expected': 'terça-feira', 'mode': 'contains'}, {'id': 'logica_028', 'category': 'logica', 'language': 'pt', 'prompt': 'Se o copo A está cheio e o B está vazio, qual está vazio?', 'expected': 'b', 'mode': 'contains'}, {'id': 'logica_029', 'category': 'logica', 'language': 'pt', 'prompt': 'Se Bia é mais nova que Ana, quem é mais velha?', 'expected': 'ana', 'mode': 'contains'}, {'id': 'logica_030', 'category': 'logica', 'language': 'pt', 'prompt': 'Qual é o próximo número: 2, 4, 6, 8, ...?', 'expected': '10', 'mode': 'contains'}, {'id': 'matematica_031', 'category': 'matematica', 'language': 'pt', 'prompt': 'Quanto é 7 + 5?', 'expected': '12', 'mode': 'number'}, {'id': 'matematica_032', 'category': 'matematica', 'language': 'pt', 'prompt': 'Quanto é 9 - 4?', 'expected': '5', 'mode': 'number'}, {'id': 'matematica_033', 'category': 'matematica', 'language': 'pt', 'prompt': 'Quanto é 6 vezes 3?', 'expected': '18', 'mode': 'number'}, {'id': 'matematica_034', 'category': 'matematica', 'language': 'pt', 'prompt': 'Quanto é 20 dividido por 4?', 'expected': '5', 'mode': 'number'}, {'id': 'matematica_035', 'category': 'matematica', 'language': 'pt', 'prompt': 'Quanto é 11 + 13?', 'expected': '24', 'mode': 'number'}, {'id': 'matematica_036', 'category': 'matematica', 'language': 'pt', 'prompt': 'Quanto é 8 vezes 7?', 'expected': '56', 'mode': 'number'}, {'id': 'matematica_037', 'category': 'matematica', 'language': 'pt', 'prompt': 'Quanto é 100 menos 37?', 'expected': '63', 'mode': 'number'}, {'id': 'matematica_038', 'category': 'matematica', 'language': 'pt', 'prompt': 'Quanto é 3 ao quadrado?', 'expected': '9', 'mode': 'number'}, {'id': 'matematica_039', 'category': 'matematica', 'language': 'pt', 'prompt': 'Quanto é 15 + 16?', 'expected': '31', 'mode': 'number'}, {'id': 'matematica_040', 'category': 'matematica', 'language': 'pt', 'prompt': 'Quanto é 2 vezes 12?', 'expected': '24', 'mode': 'number'}, {'id': 'ingles_041', 'category': 'ingles', 'language': 'en', 'prompt': 'What is the capital of Brazil?', 'expected': 'brasilia', 'mode': 'contains'}, {'id': 'ingles_042', 'category': 'ingles', 'language': 'en', 'prompt': 'Say only the word blue.', 'expected': 'blue', 'mode': 'exact'}, {'id': 'ingles_043', 'category': 'ingles', 'language': 'en', 'prompt': 'If Alice is older than Beth, who is younger?', 'expected': 'beth', 'mode': 'contains'}, {'id': 'ingles_044', 'category': 'ingles', 'language': 'en', 'prompt': 'What is 8 plus 5?', 'expected': '13', 'mode': 'contains'}, {'id': 'ingles_045', 'category': 'ingles', 'language': 'en', 'prompt': 'What is the capital of France?', 'expected': 'paris', 'mode': 'contains'}, {'id': 'ingles_046', 'category': 'ingles', 'language': 'en', 'prompt': 'Reply with exactly YES.', 'expected': 'YES', 'mode': 'exact'}, {'id': 'ingles_047', 'category': 'ingles', 'language': 'en', 'prompt': 'What is 3 times 7?', 'expected': '21', 'mode': 'contains'}, {'id': 'ingles_048', 'category': 'ingles', 'language': 'en', 'prompt': 'Which planet is called the red planet?', 'expected': 'mars', 'mode': 'contains'}, {'id': 'ingles_049', 'category': 'ingles', 'language': 'en', 'prompt': 'What is the opposite of hot?', 'expected': 'cold', 'mode': 'contains'}, {'id': 'ingles_050', 'category': 'ingles', 'language': 'en', 'prompt': 'How many days are in a week?', 'expected': '7', 'mode': 'contains'}, {'id': 'espanhol_051', 'category': 'espanhol', 'language': 'es', 'prompt': '¿Cuál es la capital de Brasil?', 'expected': 'brasilia', 'mode': 'contains'}, {'id': 'espanhol_052', 'category': 'espanhol', 'language': 'es', 'prompt': 'Di solamente la palabra azul.', 'expected': 'azul', 'mode': 'exact'}, {'id': 'espanhol_053', 'category': 'espanhol', 'language': 'es', 'prompt': 'Si Ana es mayor que Bia, ¿quién es menor?', 'expected': 'bia', 'mode': 'contains'}, {'id': 'espanhol_054', 'category': 'espanhol', 'language': 'es', 'prompt': '¿Cuánto es 8 más 5?', 'expected': '13', 'mode': 'contains'}, {'id': 'espanhol_055', 'category': 'espanhol', 'language': 'es', 'prompt': '¿Cuál es la capital de Francia?', 'expected': 'parís', 'mode': 'contains'}, {'id': 'espanhol_056', 'category': 'espanhol', 'language': 'es', 'prompt': 'Responde exactamente SÍ.', 'expected': 'SÍ', 'mode': 'exact'}, {'id': 'espanhol_057', 'category': 'espanhol', 'language': 'es', 'prompt': '¿Cuánto es 3 por 7?', 'expected': '21', 'mode': 'contains'}, {'id': 'espanhol_058', 'category': 'espanhol', 'language': 'es', 'prompt': '¿Qué planeta es conocido como planeta rojo?', 'expected': 'marte', 'mode': 'contains'}, {'id': 'espanhol_059', 'category': 'espanhol', 'language': 'es', 'prompt': '¿Cuántos días tiene una semana?', 'expected': '7', 'mode': 'contains'}, {'id': 'espanhol_060', 'category': 'espanhol', 'language': 'es', 'prompt': '¿Cuál es el opuesto de frío?', 'expected': 'caliente', 'mode': 'contains'}, {'id': 'codigo_061', 'category': 'codigo', 'language': 'pt', 'prompt': 'Em Python, escreva uma expressão que some 2 e 3.', 'expected': '2+3', 'mode': 'contains'}, {'id': 'codigo_062', 'category': 'codigo', 'language': 'pt', 'prompt': 'Qual função Python imprime uma mensagem na tela?', 'expected': 'print', 'mode': 'contains'}, {'id': 'codigo_063', 'category': 'codigo', 'language': 'pt', 'prompt': 'Em Python, qual palavra-chave define uma função?', 'expected': 'def', 'mode': 'contains'}, {'id': 'codigo_064', 'category': 'codigo', 'language': 'pt', 'prompt': 'Em Python, qual função retorna o tamanho de uma lista?', 'expected': 'len', 'mode': 'contains'}, {'id': 'codigo_065', 'category': 'codigo', 'language': 'pt', 'prompt': 'Qual é o operador de igualdade em Python?', 'expected': '==', 'mode': 'contains'}, {'id': 'parafrase_066', 'category': 'parafrase', 'language': 'pt', 'prompt': 'Se Ana é mais velha que Bia, quem é mais jovem?', 'expected': 'bia', 'mode': 'contains'}, {'id': 'parafrase_067', 'category': 'parafrase', 'language': 'pt', 'prompt': 'Ana tem mais idade que Bia. Quem tem menos idade?', 'expected': 'bia', 'mode': 'contains'}, {'id': 'parafrase_068', 'category': 'parafrase', 'language': 'pt', 'prompt': 'Entre Ana e Bia, Ana nasceu antes. Quem nasceu depois?', 'expected': 'bia', 'mode': 'contains'}, {'id': 'parafrase_069', 'category': 'parafrase', 'language': 'pt', 'prompt': 'Se Pedro é menor que João, quem é maior?', 'expected': 'joão', 'mode': 'contains'}, {'id': 'parafrase_070', 'category': 'parafrase', 'language': 'pt', 'prompt': 'João supera Pedro em altura. Quem é mais baixo?', 'expected': 'pedro', 'mode': 'contains'}, {'id': 'robustez_instrucao_071', 'category': 'robustez_instrucao', 'language': 'pt', 'prompt': 'Responda apenas azul, sem pontuação.', 'expected': 'azul', 'mode': 'exact'}, {'id': 'robustez_instrucao_072', 'category': 'robustez_instrucao', 'language': 'pt', 'prompt': 'Responda somente com a palavra azul, sem explicação.', 'expected': 'azul', 'mode': 'exact'}, {'id': 'robustez_instrucao_073', 'category': 'robustez_instrucao', 'language': 'pt', 'prompt': 'A resposta deve ser exatamente azul.', 'expected': 'azul', 'mode': 'exact'}, {'id': 'robustez_instrucao_074', 'category': 'robustez_instrucao', 'language': 'pt', 'prompt': 'Retorne apenas a palavra azul.', 'expected': 'azul', 'mode': 'exact'}, {'id': 'robustez_instrucao_075', 'category': 'robustez_instrucao', 'language': 'pt', 'prompt': 'Escreva uma única palavra: azul.', 'expected': 'azul', 'mode': 'exact'}, {'id': 'matematica_076', 'category': 'matematica', 'language': 'pt', 'prompt': 'Calcule 13 + 8.', 'expected': '21', 'mode': 'number'}, {'id': 'matematica_077', 'category': 'matematica', 'language': 'pt', 'prompt': 'Calcule 17 + 6.', 'expected': '23', 'mode': 'number'}, {'id': 'matematica_078', 'category': 'matematica', 'language': 'pt', 'prompt': 'Calcule 24 + 19.', 'expected': '43', 'mode': 'number'}, {'id': 'matematica_079', 'category': 'matematica', 'language': 'pt', 'prompt': 'Calcule 31 + 27.', 'expected': '58', 'mode': 'number'}, {'id': 'matematica_080', 'category': 'matematica', 'language': 'pt', 'prompt': 'Calcule 45 + 16.', 'expected': '61', 'mode': 'number'}, {'id': 'matematica_081', 'category': 'matematica', 'language': 'pt', 'prompt': 'Calcule 63 + 12.', 'expected': '75', 'mode': 'number'}, {'id': 'matematica_082', 'category': 'matematica', 'language': 'pt', 'prompt': 'Calcule 19 + 14.', 'expected': '33', 'mode': 'number'}, {'id': 'matematica_083', 'category': 'matematica', 'language': 'pt', 'prompt': 'Calcule 28 + 35.', 'expected': '63', 'mode': 'number'}, {'id': 'matematica_084', 'category': 'matematica', 'language': 'pt', 'prompt': 'Calcule 56 + 17.', 'expected': '73', 'mode': 'number'}, {'id': 'matematica_085', 'category': 'matematica', 'language': 'pt', 'prompt': 'Calcule 72 + 19.', 'expected': '91', 'mode': 'number'}, {'id': 'matematica_086', 'category': 'matematica', 'language': 'pt', 'prompt': 'Calcule 14 - 6.', 'expected': '8', 'mode': 'number'}, {'id': 'matematica_087', 'category': 'matematica', 'language': 'pt', 'prompt': 'Calcule 22 - 9.', 'expected': '13', 'mode': 'number'}, {'id': 'matematica_088', 'category': 'matematica', 'language': 'pt', 'prompt': 'Calcule 30 - 13.', 'expected': '17', 'mode': 'number'}, {'id': 'matematica_089', 'category': 'matematica', 'language': 'pt', 'prompt': 'Calcule 47 - 21.', 'expected': '26', 'mode': 'number'}, {'id': 'matematica_090', 'category': 'matematica', 'language': 'pt', 'prompt': 'Calcule 85 - 33.', 'expected': '52', 'mode': 'number'}, {'id': 'matematica_091', 'category': 'matematica', 'language': 'pt', 'prompt': 'Calcule 4 × 8.', 'expected': '32', 'mode': 'number'}, {'id': 'matematica_092', 'category': 'matematica', 'language': 'pt', 'prompt': 'Calcule 6 × 9.', 'expected': '54', 'mode': 'number'}, {'id': 'matematica_093', 'category': 'matematica', 'language': 'pt', 'prompt': 'Calcule 7 × 8.', 'expected': '56', 'mode': 'number'}, {'id': 'matematica_094', 'category': 'matematica', 'language': 'pt', 'prompt': 'Calcule 12 × 3.', 'expected': '36', 'mode': 'number'}, {'id': 'matematica_095', 'category': 'matematica', 'language': 'pt', 'prompt': 'Calcule 11 × 4.', 'expected': '44', 'mode': 'number'}]
print(f"{len(TESTS)} questões | categorias:",dict(Counter(t["category"] for t in TESTS)))

## 8. Executar avaliação — perfis de custo controlado

In [ ]:
PROFILE='core'  # 'smoke' (3), 'core' (~65), 'full' (todas)
TEMPERATURES=[0.0] # acrescente 0.7 apenas depois da rodada determinística
REPEATS=1 # aumente para 3 se desejar estudar estabilidade amostral
MAX_NEW_TOKENS_EVAL=56

if PROFILE=='smoke':selected=[t for t in TESTS if t['prompt'] in SMOKE]
elif PROFILE=='core':
    selected=[];counts=Counter()
    for t in TESTS:
        if counts[t['category']]<8:
            selected.append(t);counts[t['category']]+=1
elif PROFILE=='full':selected=TESTS
else:raise ValueError(PROFILE)
print('Avaliações previstas:',len(selected)*len(TEMPERATURES)*REPEATS)


## 9. Pontuação automática conservadora + diagnósticos

In [ ]:
def normalize(s):
    s=unicodedata.normalize('NFKD',str(s).casefold())
    s=''.join(c for c in s if not unicodedata.combining(c))
    return re.sub(r'\s+',' ',s).strip()
def score(t,answer):
    expected=normalize(t['expected']);actual=normalize(answer)
    if t['mode']=='exact':return int(actual==expected)
    if t['mode']=='number':
        nums=re.findall(r'(?<![\w])[-+]?\d+(?![\w])',actual)
        return int(len(nums)==1 and nums[0]==expected)
    if t['mode']=='contains':
        return int(bool(re.search(r'(?<!\w)'+re.escape(expected)+r'(?!\w)',actual)))
    raise ValueError(t['mode'])
def repetition_ratio(s):
    words=re.findall(r'\w+',normalize(s))
    if len(words)<6:return 0.0
    tri=[tuple(words[i:i+3]) for i in range(len(words)-2)]
    return round(1-len(set(tri))/len(tri),4)


In [ ]:
RESULT_DIR.mkdir(parents=True,exist_ok=True)
run_id=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
raw_path=RESULT_DIR/f'results_{run_id}.jsonl'
results=[]
with raw_path.open('w',encoding='utf-8') as f:
    for idx,t in enumerate(selected):
        for temp in TEMPERATURES:
            for rep in range(REPEATS):
                seed=SEED+rep
                try:
                    out=generate(t['prompt'],temperature=temp,max_new_tokens=MAX_NEW_TOKENS_EVAL,seed=seed)
                    row={**t,**out,'temperature':temp,'repeat':rep,'seed':seed,
                         'correct':score(t,out['answer']),'repetition_3gram':repetition_ratio(out['answer']),'error':None}
                except Exception as e:
                    row={**t,'temperature':temp,'repeat':rep,'seed':seed,'answer':'','correct':None,'error':repr(e)}
                results.append(row);f.write(json.dumps(row,ensure_ascii=False)+'\n');f.flush()
        if (idx+1)%10==0:print(f'{idx+1}/{len(selected)} questões processadas')
print('Concluído:',raw_path)


## 10. Relatório por categoria, falhas e arquivos CSV/JSON

In [ ]:
import pandas as pd
frame=pd.DataFrame(results)
valid=frame[frame['error'].isna()].copy()
summary=(valid.groupby(['category','language','temperature'],dropna=False)
         .agg(n=('correct','size'),accuracy=('correct','mean'),avg_seconds=('seconds','mean'),avg_repetition=('repetition_3gram','mean'))
         .reset_index().sort_values(['category','temperature']))
display(summary)
print('Taxa geral (questões avaliadas):',round(float(valid['correct'].mean()),4) if len(valid) else 'sem resultados')
print('Falhas de execução:',int(frame['error'].notna().sum()))
print('Amostra de respostas incorretas:')
display(valid[valid['correct']==0][['category','prompt','expected','answer']].head(35))
csv_path=RESULT_DIR/f'results_{run_id}.csv';frame.to_csv(csv_path,index=False)
summary_path=RESULT_DIR/f'summary_{run_id}.csv';summary.to_csv(summary_path,index=False)
metadata={'model':'Maia Lite 0.3.0','base_checkpoint':str(BASE_CKPT),'adapter_checkpoint':str(ADAPTER_CKPT),
          'adapter_update':FINAL_UPDATE,'profile':PROFILE,'temperatures':TEMPERATURES,'repeats':REPEATS,
          'max_new_tokens':MAX_NEW_TOKENS_EVAL,'repetition_penalty':REPETITION_PENALTY,
          'prompt_template':'Usuário: {question}\nAssistente:\n','question_count':len(selected),
          'total_generations':len(results),'torch':torch.__version__,'timestamp_utc':run_id,
          'notes':'Scores automáticos indicativos; revisar manualmente; nenhuma conversão ou treinamento'}
meta_path=RESULT_DIR/f'metadata_{run_id}.json';meta_path.write_text(json.dumps(metadata,indent=2,ensure_ascii=False))
print('Arquivos:',raw_path,csv_path,summary_path,meta_path,sep='\n')


## 11. Revisão humana e diagnóstico de formato de prompt (opcional)

In [ ]:
# Inspecione todos os casos, inclusive acertos automáticos que podem ser falsos positivos.
for r in results[:min(20,len(results))]:
    print(f"[{r['category']}] {r['prompt']}\nEsperado: {r['expected']}\nObtido: {r['answer']}\nAcerto automático: {r['correct']}\n")


### Interpretação

- **Falha de obediência** (por exemplo, resposta diferente de `azul`) não é resolvida por uma loss de validação menor.
- **Raciocínio** deve ser interpretado por tarefa e por paráfrase, não apenas pela média.
- **Código**: a pontuação por palavra-chave é insuficiente; revisar manualmente ou executar testes isolados.
- **Comparações** com outros modelos exigem mesmo banco de provas, prompt adequado a cada modelo e condições de geração documentadas.
- Este notebook **não faz exportação GGUF/Ollama**. Isso ficará em um notebook separado, depois da auditoria.
